# VN Address Intelligence - Ablation Study (N=1000)

Chạy 5 cấu hình Ablation Study trên Google Colab với GPU T4.

## Cấu hình

- **A1:** NER + mGTE + LLM (Full)
- **A2:** NER + mGTE (No LLM)
- **A3:** mGTE only (Retrieval-only)
- **A4:** NER + LLM (No Retrieval)
- **A5:** NER + PhoBERT Siamese + LLM

## Thời gian ước tính

- Mỗi cấu hình: ~20–30 phút
- Tổng: ~2–3 giờ

In [ ]:
# === CELL 1: Setup & Mount Drive ===
from google.colab import drive
import sys

# Mount Google Drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q torch transformers sentence-transformers pandas tqdm seqeval

# Copy files from Drive
!cp /content/drive/MyDrive/colab_vnai/ground_truth.db /content/
!cp /content/drive/MyDrive/colab_vnai/noise_profiles.json /content/

# Clone repo (hoặc copy code)
!git clone https://github.com/YOUR_REPO/vn-address-intelligence.git
sys.path.insert(0, '/content/vn-address-intelligence/src')

print("✓ Setup completed")

In [ ]:
# === CELL 2: Load Models (GPU) ===
import torch
from app.ai.ner_model import NERModel
from app.ai.siamese_mgte import SiameseMGTE
from app.ai.llm_qwen import LLMQwen3

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Load NER
print("\nLoading NER model...")
ner = NERModel(device="cuda")
print("✓ NER loaded")

# Load mGTE retriever
print("\nLoading mGTE retriever...")
retriever = SiameseMGTE(corpus_limit=12000, device="cuda")
print("✓ mGTE loaded")

# Load LLM
print("\nLoading LLM (Qwen 4-bit)...")
llm = LLMQwen3(quantization="4bit", device="cuda")
print("✓ LLM loaded")

print("\n" + "="*60)
print("All models loaded successfully!")
print("="*60)

In [ ]:
# === CELL 3: Extract Cohort & Apply Noise ===
import sqlite3
import json
import random
import re
from typing import Dict, Any

# Load noise profiles
with open("/content/noise_profiles.json", "r", encoding="utf-8") as f:
    NOISE_PROFILES = json.load(f)

def apply_noise(address: str, profile_name: str = "SUP-1.0.0") -> str:
    """Apply noise theo profile."""
    profile = NOISE_PROFILES[profile_name]
    result = address
    
    # Remove diacritics
    if random.random() < profile.get("remove_diacritics_prob", 0):
        import unicodedata
        result = unicodedata.normalize('NFD', result)
        result = ''.join(c for c in result if unicodedata.category(c) != 'Mn')
    
    # Abbreviate
    if random.random() < profile.get("abbreviate_prob", 0):
        abbr_map = profile.get("abbreviations", {})
        for full, abbr in abbr_map.items():
            result = re.sub(rf"\b{full}\b", abbr, result, flags=re.IGNORECASE)
    
    # Typos
    if random.random() < profile.get("typo_prob", 0):
        chars = list(result)
        if len(chars) > 5:
            idx = random.randint(1, len(chars) - 2)
            chars[idx], chars[idx + 1] = chars[idx + 1], chars[idx]
        result = ''.join(chars)
    
    return result

# Extract cohort
print("Extracting cohort from SQLite...")
conn = sqlite3.connect("/content/ground_truth.db")
conn.row_factory = sqlite3.Row
cur = conn.cursor()
cur.execute("SELECT * FROM ground_truth")
all_rows = cur.fetchall()
conn.close()

print(f"✓ Loaded {len(all_rows)} rows from ground_truth")

# Sample N=1000 with seed=3001
random.seed(3001)
cohort_rows = random.sample(all_rows, 1000)

# Apply noise
specimens = []
for row in cohort_rows:
    noisy = apply_noise(row['address'], "SUP-1.0.0")
    specimens.append({
        'gt_id': row['id'],
        'raw_address': noisy,
        'ref_address_v2': row['address'],
        'province_id': row['province_id'],
        'district_id': row['district_id'],
        'ward_id': row['ward_id']
    })

print(f"✓ Created {len(specimens)} specimens with noise")
print(f"\nExample:")
print(f"  Original: {specimens[0]['ref_address_v2']}")
print(f"  Noisy:    {specimens[0]['raw_address']}")

In [ ]:
# === CELL 4: Run A1 (NER + mGTE + LLM) ===
import time
from tqdm import tqdm

print("="*60)
print("Running A1: NER + mGTE + LLM (Full)")
print("="*60)

results_a1 = []

for spec in tqdm(specimens, desc="A1 Progress"):
    start = time.time()
    
    try:
        # Step 1: NER
        ner_entities = ner.predict(spec['raw_address'])
        
        # Step 2: Retrieval
        candidates = retriever.retrieve_top_k(spec['raw_address'], k=5)
        
        # Step 3: LLM refinement
        pred_standardized = llm.refine(
            raw=spec['raw_address'],
            ner_entities=ner_entities,
            candidates=candidates
        )
        
        latency_ms = (time.time() - start) * 1000
        
        results_a1.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': pred_standardized,
            'latency_ms': latency_ms,
            'config': 'A1_FULL'
        })
    except Exception as e:
        print(f"\nError on specimen {spec['gt_id']}: {e}")
        results_a1.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': None,
            'latency_ms': None,
            'config': 'A1_FULL'
        })

avg_latency = sum(r['latency_ms'] for r in results_a1 if r['latency_ms']) / len([r for r in results_a1 if r['latency_ms']])
print(f"\n✓ A1 completed: {len(results_a1)} predictions")
print(f"  Avg latency: {avg_latency:.1f} ms")

In [ ]:
# === CELL 5: Run A2 (NER + mGTE, No LLM) ===
print("="*60)
print("Running A2: NER + mGTE (No LLM)")
print("="*60)

results_a2 = []

for spec in tqdm(specimens, desc="A2 Progress"):
    start = time.time()
    
    try:
        # NER + Retrieval, pick top-1
        ner_entities = ner.predict(spec['raw_address'])
        candidates = retriever.retrieve_top_k(spec['raw_address'], k=5)
        pred_standardized = candidates[0] if candidates else spec['raw_address']
        
        latency_ms = (time.time() - start) * 1000
        
        results_a2.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': pred_standardized,
            'latency_ms': latency_ms,
            'config': 'A2_NER_MGTE'
        })
    except Exception as e:
        print(f"\nError: {e}")
        results_a2.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': None,
            'latency_ms': None,
            'config': 'A2_NER_MGTE'
        })

avg_latency = sum(r['latency_ms'] for r in results_a2 if r['latency_ms']) / len([r for r in results_a2 if r['latency_ms']])
print(f"\n✓ A2 completed: {len(results_a2)} predictions")
print(f"  Avg latency: {avg_latency:.1f} ms")

In [ ]:
# === CELL 6: Run A3 (mGTE only) ===
print("="*60)
print("Running A3: mGTE only (Retrieval-only)")
print("="*60)

results_a3 = []

for spec in tqdm(specimens, desc="A3 Progress"):
    start = time.time()
    
    try:
        # Retrieval only
        candidates = retriever.retrieve_top_k(spec['raw_address'], k=5)
        pred_standardized = candidates[0] if candidates else spec['raw_address']
        
        latency_ms = (time.time() - start) * 1000
        
        results_a3.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': pred_standardized,
            'latency_ms': latency_ms,
            'config': 'A3_MGTE_ONLY'
        })
    except Exception as e:
        print(f"\nError: {e}")
        results_a3.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': None,
            'latency_ms': None,
            'config': 'A3_MGTE_ONLY'
        })

avg_latency = sum(r['latency_ms'] for r in results_a3 if r['latency_ms']) / len([r for r in results_a3 if r['latency_ms']])
print(f"\n✓ A3 completed: {len(results_a3)} predictions")
print(f"  Avg latency: {avg_latency:.1f} ms")

In [ ]:
# === CELL 7: Run A4 (NER + LLM, No Retrieval) ===
print("="*60)
print("Running A4: NER + LLM (No Retrieval)")
print("="*60)

results_a4 = []

for spec in tqdm(specimens, desc="A4 Progress"):
    start = time.time()
    
    try:
        # NER + LLM (no retrieval candidates)
        ner_entities = ner.predict(spec['raw_address'])
        pred_standardized = llm.refine(
            raw=spec['raw_address'],
            ner_entities=ner_entities,
            candidates=[]  # No retrieval
        )
        
        latency_ms = (time.time() - start) * 1000
        
        results_a4.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': pred_standardized,
            'latency_ms': latency_ms,
            'config': 'A4_NER_LLM'
        })
    except Exception as e:
        print(f"\nError: {e}")
        results_a4.append({
            'gt_id': spec['gt_id'],
            'raw_address': spec['raw_address'],
            'ref_address_v2': spec['ref_address_v2'],
            'pred_standardized': None,
            'latency_ms': None,
            'config': 'A4_NER_LLM'
        })

avg_latency = sum(r['latency_ms'] for r in results_a4 if r['latency_ms']) / len([r for r in results_a4 if r['latency_ms']])
print(f"\n✓ A4 completed: {len(results_a4)} predictions")
print(f"  Avg latency: {avg_latency:.1f} ms")

In [ ]:
# === CELL 8: Export Results ===
import pandas as pd
from google.colab import files

# Combine all results
all_results = results_a1 + results_a2 + results_a3 + results_a4

df = pd.DataFrame(all_results)
df.to_csv("ablation_n1000_results.csv", index=False)

print("="*60)
print("EXPORT SUMMARY")
print("="*60)
print(f"Total rows: {len(df)}")
print(f"\nBy config:")
print(df.groupby('config').agg({
    'latency_ms': ['count', 'mean', 'std']
}))

print(f"\n✓ Exported to ablation_n1000_results.csv")
print(f"\nDownloading...")
files.download("ablation_n1000_results.csv")